# Chinese BERT 
This example demonstrates chinese-roberta-wwm-ext.

## 1. Translation

In [ ]:
# this cell downloads via a mirror (useful in mainland China)

import os
from transformers import MarianTokenizer, MarianMTModel

# 1. set the mirror endpoint before any transformers call
#    it only applies to the current session
print("Setting the Hugging Face mirror...")
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print("Mirror set.")

# 2. now run the usual download-and-save code
model_name = "Helsinki-NLP/opus-mt-en-zh"


print(f"\nDownloading tokenizer from the mirror: {model_name}")
tokenizer = MarianTokenizer.from_pretrained(model_name)
print("Tokenizer downloaded.")

print(f"\nDownloading model from the mirror: {model_name} (this may take a while)...")
model = MarianMTModel.from_pretrained(model_name)
print("Model downloaded!")

# local save path
local_path = "./model/opus-mt-en-zh"

# save the model and tokenizer locally
print(f"\nSaving model and tokenizer to: {local_path}...")
tokenizer.save_pretrained(local_path)
model.save_pretrained(local_path)

print(f"\nDone! Model and tokenizer saved to: {local_path}")


In [ ]:
# if you're outside mainland China, use this cell to download the model
from transformers import MarianTokenizer, MarianMTModel

# model name
model_name = "Helsinki-NLP/opus-mt-en-zh"

# download the model and tokenizer
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# # fallback: load from safetensors weights
# model = MarianMTModel.from_pretrained(model_name, use_safetensors=True)

# local save path
local_path = "./model/opus-mt-en-zh"

# save the model and tokenizer locally
tokenizer.save_pretrained(local_path)
model.save_pretrained(local_path)

print(f"Model and tokenizer saved to: {local_path}")

In [ ]:
from transformers import MarianTokenizer, MarianMTModel

# local path
local_path = "./model/opus-mt-en-zh"

# load tokenizer and model from local files
tokenizer = MarianTokenizer.from_pretrained(local_path)
model = MarianMTModel.from_pretrained(local_path)

print("Model and tokenizer loaded from local files")

In [ ]:
# load the file contents
import time

with open('data/FinancialPhraseBank/eng_sent.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# split into lines
lines = content.split('\n')
translated_lines = []

# record translation start time
start_time = time.time()

for line in lines:
    if line.strip():
        tokenized_text = tokenizer.encode(line, return_tensors="pt", max_length=512, truncation=True)
        translated = model.generate(tokenized_text)
        translated_text = tokenizer.decode(translated[0], skip_special_tokens=True)
        translated_lines.append(translated_text)

# record translation end time
end_time = time.time()
tran_time = end_time - start_time  # elapsed time
print(f"Translation time: {tran_time:.2f} seconds")

# save the results
with open('data/FinancialPhraseBank/chn_sent.txt', 'w', encoding='utf-8') as output_file:
    output_file.write('\n'.join(translated_lines))

print("Translation done; saved to chn_sent.txt")

In [ ]:
translated_lines

## 2. Chinese BERT

### 2.1 Download / load the local model

In [ ]:
# download Chinese BERT (chinese-roberta-wwm-ext) locally
from transformers import AutoTokenizer, AutoModel

# model name
model_name = "hfl/chinese-roberta-wwm-ext"

# download and save the tokenizer and model locally
save_directory = "./model/chinese-roberta-wwm-ext"

# download the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.save_pretrained(save_directory)

# download the model
# model = AutoModel.from_pretrained(model_name)
# fallback: load from safetensors weights
model = AutoModel.from_pretrained(model_name, use_safetensors=True)
model.save_pretrained(save_directory)

print(f"Model and tokenizer saved to {save_directory}")

In [ ]:
from transformers import AutoTokenizer, AutoModel

# local path
local_directory = "./model/chinese-roberta-wwm-ext"

# load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_directory)

# load the model
model = AutoModel.from_pretrained(local_directory)

print("Model and tokenizer loaded successfully from local path.")

Model and tokenizer loaded successfully from local path.


### 2.2 Load data and tokenize

In [ ]:
with open('data/FinancialPhraseBank/chn_sent.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# split into lines
FPB_data_raw = content.split('\n')

In [ ]:
import json
FPB_label_raw = json.load(open('data/FinancialPhraseBank/labels.json'))

In [ ]:
len(FPB_data_raw)==len(FPB_label_raw)

True

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_scheduler

import evaluate

In [ ]:
# tokenize and encode the sentences with BERT's tokenizer
encodings = tokenizer(
    FPB_data_raw,                  # list of input sentences
    truncation=True,            # truncate sentences longer than max_length
    padding=True,               # pad shorter sentences to a uniform length
    max_length=128,             # maximum sentence length; the rest is truncated
    return_tensors="pt"         # return PyTorch tensors
)

# get the encoded input_ids and attention_mask
input_ids = encodings["input_ids"]               # encoded token ids (like the int ids used with word2vec)
attention_mask = encodings["attention_mask"]     # attention mask: distinguishes real tokens from padding
labels = torch.tensor(FPB_label_raw)                    # convert labels to a PyTorch tensor

# create the dataset and data loader
# wrap input_ids, attention_mask and labels in a TensorDataset
dataset = torch.utils.data.TensorDataset(input_ids, attention_mask, labels)

### 2.3 Train/Val/Test Split

In [ ]:
from torch.utils.data import random_split, DataLoader
# set the random seed
torch.manual_seed(42)

# split ratios
train_size = int(0.8 * len(dataset))  # 80% train
val_size = int(0.1 * len(dataset))    # 10% validation
test_size = len(dataset) - train_size - val_size  # remaining 10% test

# split the dataset by ratio
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)

# create the data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)  # training loader
val_loader = DataLoader(val_dataset, batch_size=16, drop_last=True)                   # validation loader
test_loader = DataLoader(test_dataset, batch_size=16, drop_last=True)                 # test loader

# check the split
print(f"Train size: {len(train_dataset)}, Validation size: {len(val_dataset)}, Test size: {len(test_dataset)}")

Train size: 4573, Validation size: 571, Test size: 573


### 2.4 Chinese BERT 3-way Classification

In [ ]:
# load the local pretrained BERT, ignoring the mismatched classification head
model_local = BertForSequenceClassification.from_pretrained(
    "./model/chinese-roberta-wwm-ext",  # local model path
    num_labels=3,                       # number of labels (3-way classification)
    ignore_mismatched_sizes=True        # ignore size mismatches
)

In [ ]:
optimizer = optim.AdamW(model_local.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 5  # assume 5 epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # choose device: GPU (CUDA) if available, else CPU
print(device)

cpu


In [ ]:
# load the evaluation metric
metric = evaluate.load("accuracy")
model_local.to(device)
# training and validation
epochs = 3
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")

    # record training start time
    start_time = time.time()

    # train the model
    model_local.train()  # set the model to training mode
    for batch in train_loader:
        # move the batch to GPU or CPU
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        
        # forward pass: compute loss and logits
        outputs = model_local(**batch)
        loss = outputs.loss          # loss for this batch
        logits = outputs.logits      # model predictions (logits)

        # backpropagation and parameter update
        optimizer.zero_grad()        # clear the optimizer gradients
        loss.backward()              # compute gradients
        optimizer.step()             # update model parameters
        scheduler.step()             # update the learning rate

    # record training end time
    end_time = time.time()
    train_time = end_time - start_time  # elapsed time
    print(f"Training time: {train_time:.2f} seconds")

    # validate the model
    model_local.eval()  # set the model to eval mode

    # record validation start time
    start_time = time.time()

    for batch in val_loader:
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        with torch.no_grad():  # disable gradients to save memory and speed up
            outputs = model_local(**batch)

        logits = outputs.logits                # get the logits
        predictions = torch.argmax(logits, dim=-1)  # predicted class
        metric.add_batch(predictions=predictions, references=batch["labels"])  # update the metric

    # record validation end time
    end_time = time.time()
    val_time = end_time - start_time  # elapsed time
    print(f"Validation time: {val_time:.2f} seconds")

    # compute and print the accuracy
    val_accuracy = metric.compute()["accuracy"]
    print(f"Epoch {epoch + 1}: Validation Accuracy = {val_accuracy:.4f}")

Epoch 1/3
Training time: 416.00 seconds
Validation time: 15.36 seconds
Epoch 1: Validation Accuracy = 0.9429
Epoch 2/3
Training time: 454.68 seconds
Validation time: 15.25 seconds
Epoch 2: Validation Accuracy = 0.9393
Epoch 3/3
Training time: 461.34 seconds
Validation time: 15.45 seconds
Epoch 3: Validation Accuracy = 0.9500
